In [16]:
import numpy as np 

from scipy.constants import c as c0
from scipy.interpolate import CubicSpline

In [ ]:
import sys
from pathlib import Path

# Adding to path
PROJECT_ROOT = Path("/home/lsito/mast")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.gui.meas_config_dialog import MeasurementConfigDialog
from src.data_models.meas_config import MeasurementConfig

from src.gui.structure_config_dialog import RFStructureLoaderDialog
from src.data_models.rf_structure import RFStructureParams

from src.core.config_parser import DataMixin

In [ ]:
# In case you need to check the current working directory
from pathlib import Path
Path.cwd()

PosixPath('/home/lsito/mast/notebook')

In [42]:
RF_params = RFStructureParams.from_json("../data/2021_03_18_-_TD31_Nx-raw_data/00-Config_Files/TD31_R1_CC.json")
Meas_params = MeasurementConfig()

print(RF_params.vg)
print(RF_params.noc)
print(RF_params.phi0)
print(RF_params.Q0)
print(Meas_params.target_frequency_mhz)

[8751932.09218, 8503739.639156, 8259142.530828, 8018140.767196, 7780734.34826, 7546923.27402, 7316707.544476, 7090087.159628, 6867062.119476, 6647632.42402, 6431798.07326, 6219569.067196, 6010935.405828, 5805897.089156, 5604454.11718, 5406606.4899, 5212354.207316, 5021697.269428, 4834635.676236, 4651169.42774, 4471298.52394, 4295022.964836, 4122342.750428, 3953257.880716, 3787768.3557, 3625874.17538, 3467575.339756, 3312871.848828, 3161763.702596, 3014250.90106, 2870333.44422]
33
2.0943951023931953
[5158.89, 5171.7468, 5184.7752, 5197.9752, 5211.3468, 5224.89, 5238.6048, 5252.4912, 5266.5492, 5280.7788, 5295.18, 5309.7528, 5324.4972, 5339.4132, 5354.5008, 5369.76, 5385.1908, 5400.7932, 5416.5672, 5432.5128, 5448.63, 5464.9188, 5481.3792, 5498.0112, 5514.8148, 5531.79, 5548.9368, 5566.2552, 5583.7452, 5601.4068, 5619.24]
11995.491004


In [48]:
# Target Frequency (Should be the frequency of the source) in Hz --> This is computed from the measurement config as target frequency to tune
f1 = 11.9942e9 
f1 = Meas_params.target_frequency_mhz * 1e6

# Resonant frequency of the bead-pull (written in the name of the file) --> Should be the frequency of the cavity
f0 = 11.993e9 

f = f0 # Just supoport variable

# Normalized frequency detuning for small detunings
    # Note: this is Eq. (3) however, it looks like the sign is opposite.
DeltaF = -2 * (f1-f0) / f0

# Group velocity interpolation
    # This is a list of group velocities for each cell, given in the json config file, apparently at the beginning of the cell (0, 1, 2, ..., noc-1)
    # we need vg at the middle of the cell (0.5, 1.5, ..., noc-0.5) for the calculation of Qe and beta, so we need to interpolate it with a cubic spline
    # 0.5   1   1.5   2   2.5   3   ...
    #  |    *    |    *    |    *
    #      vg1       vg2       vg3
vg = RF_params.vg 

    # Number of cells, given in the json config file
noc = RF_params.noc 

# Issue here
x = np.arange(0, noc)          # 1, 2, ..., noc # This is Matlab indexing
x_interp = np.arange(0.5, noc + 0.5) # 0.5, 1.5, ..., noc + 0.5
print(len(x))
print(len(x_interp))
print(len(vg))
spline = CubicSpline(x, vg)
vg_ = spline(x_interp)

# External Q approximated from Eq. (5)

    # This is the phase advance (NOT per cell), given in the json config file
phi0 = RF_params.phi0 
Q0 = RF_params.Q0
Qe = c0 * phi0 / vg_ 

# Coupling coefficient beta of each cell from Wangler Chapt.5.5
# Qe:      Qe1      Qe2      Qe3      Qe4      Qe5
#           |        |        |        |        |
# 
# beta:   beta1    beta2    beta3    beta4
#         uses     uses     uses     uses
#         Qe1,Qe2  Qe2,Qe3  Qe3,Qe4  Qe4,Qe5
# Notice that here the cell is treated as a single input system; hence the power flowing out of the second 
# port is accounted for as power lost
# Hence Beta(i) = P_ext(1)/(P0(i)+P_ext(i+1)) = 1/Qe(i) / (1/Q0(i) + 1/Qe(i+1))

Qe_i = Qe[0:-1]
Qe_ip1 = Qe[1:]

beta = (1/Qe_i) / (1/Q0 + 1/Qe_ip1)

# Reflection from the input port (from Eq. (1) with Q0 = Qe)

gamma_num = (beta-1)*(beta+1) - Qe_i**2 * DeltaF**2 - 1j*2*beta*Qe_i*DeltaF
gamma_den = (beta+1)**2 + Qe_i**2 * DeltaF**2
gamma = gamma_num / gamma_den

33
33
31


ValueError: The length of `y` along `axis`=0 doesn't match the length of `x`